# Tugas Praktikum #4 — Asyncio (Pertemuan 7)

## Persiapan: import dan fungsi dasar

Fungsi `ambil_metadata_paket()` sama persis dengan yang dipakai di materi (bagian 6.1). Kita pakai ulang di Soal 1.

In [ ]:
import asyncio
import math
import random
import statistics
import time

import aiohttp
import matplotlib.pyplot as plt


async def ambil_metadata_paket(session, nama_paket):
    url = f"https://pypi.org/pypi/{nama_paket}/json"
    async with session.get(url) as response:
        data = await response.json()
        info = data["info"]
        return {
            "nama": info["name"],
            "versi": info["version"],
            "summary": (info["summary"] or "")[:60],
        }

---
## Soal 1 — Eksperimen Skala

**Perintah:** Perbesar `daftar_paket_banyak` menjadi 20–30 paket PyPI, lalu bandingkan waktu total untuk `BATAS_KONKURENSI` = 1, 3, 5, 10, 20. Pada titik mana penambahan batas mulai memberi manfaat sangat kecil? Kaitkan dengan *diminishing returns* dari Hukum Amdahl.

1. Siapkan 24 nama paket populer.
2. Buat fungsi `jalankan_eksperimen(batas)` yang mengukur waktu total dengan `asyncio.Semaphore(batas)`.
3. Lakukan satu kali *warm-up* (hasilnya dibuang) supaya biaya DNS/TLS pertama tidak mengotori pengukuran.
4. Ulangi tiap batas 3 kali dan ambil **median**, karena latensi jaringan nyata bisa berfluktuasi.

In [ ]:
daftar_paket_banyak = [
    "requests", "numpy", "flask", "aiohttp", "pandas", "django",
    "scipy", "matplotlib", "pillow", "sqlalchemy", "pytest", "fastapi",
    "httpx", "beautifulsoup4", "click", "jinja2", "pydantic", "redis",
    "urllib3", "certifi", "idna", "rich", "tqdm", "pyyaml",
]
print(f"Jumlah paket: {len(daftar_paket_banyak)}")


async def ambil_dengan_batas(session, nama_paket, semaphore):
    async with semaphore:  # hanya N coroutine yang boleh masuk blok ini bersamaan
        return await ambil_metadata_paket(session, nama_paket)


async def jalankan_eksperimen(batas):
    """Mengambil semua paket dengan maksimal `batas` request bersamaan.
    Mengembalikan (waktu_total, jumlah_gagal)."""
    semaphore = asyncio.Semaphore(batas)
    start = time.perf_counter()
    async with aiohttp.ClientSession() as session:
        hasil = await asyncio.gather(
            *[ambil_dengan_batas(session, p, semaphore) for p in daftar_paket_banyak],
            return_exceptions=True,  # satu paket gagal tidak menggagalkan seluruh percobaan
        )
    waktu = time.perf_counter() - start
    gagal = sum(1 for h in hasil if isinstance(h, Exception))
    return waktu, gagal

Sekarang jalankan percobaan. Tiap batas diulang 3 kali dan diambil nilai **median**-nya.

In [ ]:
batas_uji = [1, 3, 5, 10, 20]
ULANGAN = 3

# Warm-up: hasilnya dibuang (DNS lookup + TLS handshake pertama biasanya lebih lambat)
await jalankan_eksperimen(5)

waktu_median = []
total_gagal = 0
for batas in batas_uji:
    percobaan = []
    for _ in range(ULANGAN):
        waktu, gagal = await jalankan_eksperimen(batas)
        percobaan.append(waktu)
        total_gagal += gagal
    waktu_median.append(statistics.median(percobaan))

speedup = [waktu_median[0] / w for w in waktu_median]

print(f"{'Batas':>6} | {'Waktu (detik)':>13} | {'Speedup':>8} | {'Peningkatan vs batas sebelumnya':>31}")
print("-" * 70)
for i, batas in enumerate(batas_uji):
    if i == 0:
        perubahan = "-"
    else:
        perubahan = f"{(waktu_median[i-1] - waktu_median[i]) / waktu_median[i-1] * 100:.0f}% lebih cepat"
    print(f"{batas:>6} | {waktu_median[i]:>13.2f} | {speedup[i]:>7.2f}x | {perubahan:>31}")
print(f"\nTotal request gagal selama percobaan: {total_gagal}")

### Mencari titik *diminishing returns* dan membandingkannya dengan Hukum Amdahl

Hukum Amdahl: jika fraksi `p` dari pekerjaan bisa dikonkurenkan dan sisanya `(1 − p)` tetap serial, maka speedup dengan `N` unit konkuren adalah

$$S(N) = \frac{1}{(1 - p) + \frac{p}{N}}$$

Kita **perkirakan `p` dari data kita sendiri** (memakai batas terbesar), lalu gambar kurva Amdahl bersama kurva ideal dan kurva hasil pengukuran. Ini hanya estimasi kasar, tetapi cukup untuk melihat polanya.

In [ ]:
# 1) Titik diminishing returns: batas pertama yang menambah kecepatan < 15% dibanding batas sebelumnya
AMBANG = 0.15
titik = None
for i in range(1, len(batas_uji)):
    peningkatan = (waktu_median[i-1] - waktu_median[i]) / waktu_median[i-1]
    if peningkatan < AMBANG:
        titik = batas_uji[i]
        break

if titik:
    print(f"Titik diminishing returns: mulai batas = {titik} "
          f"(penambahan batas hanya memberi < {AMBANG*100:.0f}% peningkatan)")
else:
    print("Belum terlihat diminishing returns yang jelas pada rentang batas yang diuji.")

# 2) Estimasi fraksi paralel p dari speedup pada batas terbesar:  S = 1 / ((1-p) + p/N)
N_maks, S_maks = batas_uji[-1], speedup[-1]
p_est = (1 - 1 / S_maks) / (1 - 1 / N_maks) if S_maks > 1 else 0.0
p_est = min(max(p_est, 0.0), 0.999)
print(f"Estimasi fraksi yang bisa dikonkurenkan (p) ~ {p_est:.2f}")
print(f"Batas speedup teoritis (N -> tak hingga)  ~ {1 / (1 - p_est):.1f}x")

# 3) Plot
def amdahl(p, n):
    return 1 / ((1 - p) + p / n)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(batas_uji, waktu_median, "o-")
ax1.set_xlabel("BATAS_KONKURENSI")
ax1.set_ylabel("Waktu total (detik)")
ax1.set_title("Waktu vs batas konkurensi")
ax1.set_xticks(batas_uji)
ax1.grid(alpha=0.3)

ax2.plot(batas_uji, speedup, "o-", label="Hasil pengukuran")
ax2.plot(batas_uji, [amdahl(p_est, n) for n in batas_uji], "s--", label=f"Amdahl (p={p_est:.2f})")
ax2.plot(batas_uji, batas_uji, ":", label="Ideal (linear)")
ax2.set_xlabel("BATAS_KONKURENSI")
ax2.set_ylabel("Speedup terhadap batas = 1")
ax2.set_title("Speedup vs batas konkurensi")
ax2.set_xticks(batas_uji)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Analisis Soal 1

- **Batas = 1** pada dasarnya sama dengan menjalankan request satu per satu: waktunya ≈ jumlah paket × latensi satu request.
- **Menaikkan batas dari 1 → 3 → 5** memberi penurunan waktu paling besar, karena sebagian besar waktu sebuah request adalah *menunggu jaringan* (I/O), dan waktu tunggu itu bisa ditumpuk.
- **Setelah batas sekitar 5–10**, grafik waktu mulai mendatar. Titik pastinya bisa dilihat pada output *"Titik diminishing returns"* di atas (angkanya bisa berbeda tiap run karena latensi jaringan nyata).
- **Batas 20 vs jumlah paket 24**: ketika batas hampir sama dengan jumlah paket, hampir semua request sudah jalan sekaligus, sehingga menaikkan batas lagi praktis tidak mengubah apa-apa. Semaphore tidak lagi menjadi pembatas.

**Kaitan dengan Hukum Amdahl.** Tidak seluruh pekerjaan bisa dikonkurenkan. Ada bagian serial `(1 − p)` yang tetap sama berapa pun batasnya, misalnya:
1. Membuat `ClientSession` dan membuka koneksi pertama (DNS + TLS handshake).
2. `response.json()` dan pengolahan hasil, yang dijalankan CPU di *satu* thread event loop.
3. Batas kapasitas di sisi server/CDN PyPI dan bandwidth jaringan kita sendiri.

Makin besar `N`, bagian `p/N` makin kecil dan bagian serial `(1 − p)` makin dominan, sehingga speedup mendekati batas `1 / (1 − p)` dan tambahan batas memberi hasil yang makin kecil. Sebagai ilustrasi, bila `p = 0,9`: `N=3` → 2,5×, `N=5` → 3,6×, `N=10` → 5,3×, `N=20` → 6,9×, dan batas teoritisnya 10×. Dari 10 ke 20 kita menggandakan batas, tetapi speedup hanya naik sekitar 1,3×. Itulah *diminishing returns*.

**Kesimpulan praktis:** batas konkurensi yang "cukup" biasanya jauh di bawah jumlah total tugas. Memilih batas sedang (misalnya 5–10) sudah mengambil hampir semua manfaat sekaligus bersikap sopan kepada server tujuan.

---
## Soal 2 — Deteksi Blocking Call

**Perintah:** Buat 3 coroutine berbeda: (1) memakai `asyncio.sleep()`, (2) memakai `time.sleep()`, (3) operasi CPU-bound murni tanpa `await`. Jalankan ketiganya dengan `asyncio.gather()` bersama satu coroutine `asyncio.sleep()` lain sebagai **"pengukur"**. Amati coroutine mana yang mengganggu pengukur, dan jelaskan mengapa.

**Ide percobaan:** coroutine `pengukur` berulang kali melakukan `await asyncio.sleep(0.1)` dan mencatat selisih waktu antar "tick". Kalau event loop sehat, selisihnya ≈ 0,1 detik. Kalau ada coroutine lain yang **memblokir** event loop, tick pengukur terlambat sebesar lama pemblokiran itu.

Supaya mudah dibaca, ketiga coroutine dijadwalkan di waktu yang berbeda (lewat `await asyncio.sleep()` di awal), sehingga jendela waktunya tidak saling tumpang tindih.

In [ ]:
T0 = time.perf_counter()          # acuan waktu bersama
jendela = {}                      # nama -> (waktu_mulai, waktu_selesai), dalam detik sejak T0


def sekarang():
    return time.perf_counter() - T0


def faktorial_manual(n):
    """CPU-bound murni: loop biasa tanpa satu pun await."""
    hasil = 1
    for i in range(2, n + 1):
        hasil *= i
    return hasil


async def pengukur(durasi_total=3.5, interval=0.1):
    """Mencatat (waktu_tick, jeda_sejak_tick_sebelumnya). Idealnya jeda ~ interval."""
    tick = []
    terakhir = sekarang()
    while sekarang() < durasi_total:
        await asyncio.sleep(interval)
        t = sekarang()
        tick.append((t, t - terakhir))
        terakhir = t
    return tick


async def coroutine_async_sleep():
    mulai = sekarang()
    await asyncio.sleep(0.6)               # NON-blocking: menyerahkan giliran ke event loop
    jendela["1. asyncio.sleep"] = (mulai, sekarang())


async def coroutine_time_sleep():
    await asyncio.sleep(0.8)               # tunggu dulu supaya jendelanya terpisah
    mulai = sekarang()
    time.sleep(0.6)                        # BLOCKING: menahan satu-satunya thread event loop
    jendela["2. time.sleep"] = (mulai, sekarang())


async def coroutine_cpu_bound():
    await asyncio.sleep(1.8)               # tunggu dulu supaya jendelanya terpisah
    mulai = sekarang()
    faktorial_manual(60000)                # CPU-bound: tidak ada await sama sekali di dalamnya
    jendela["3. CPU-bound"] = (mulai, sekarang())


T0 = time.perf_counter()
hasil_tick, *_ = await asyncio.gather(
    pengukur(),
    coroutine_async_sleep(),
    coroutine_time_sleep(),
    coroutine_cpu_bound(),
)

print(f"Jumlah tick pengukur: {len(hasil_tick)} (ideal untuk 3,5 detik = sekitar 35 tick)\n")

print("Tick yang TERLAMBAT (jeda > 0,25 detik; seharusnya sekitar 0,1 detik):")
terlambat = [(t, j) for t, j in hasil_tick if j > 0.25]
for t, j in terlambat:
    print(f"  t = {t:4.2f} s  | jeda = {j:4.2f} s")
if not terlambat:
    print("  (tidak ada)")

print("\nJeda terbesar yang dialami pengukur selama tiap coroutine berjalan:")
for nama, (mulai, selesai) in sorted(jendela.items()):
    jeda_dalam_jendela = [j for t, j in hasil_tick if (t - j) < selesai and t > mulai]
    jeda_maks = max(jeda_dalam_jendela) if jeda_dalam_jendela else 0
    status = "memblokir event loop!" if jeda_maks > 0.25 else "aman"
    print(f"  {nama:<18} | lama = {selesai - mulai:4.2f} s | jeda maks pengukur = {jeda_maks:4.2f} s -> {status}")

### Analisis Soal 2

| Coroutine | Efek pada pengukur | Penyebab |
|---|---|---|
| 1. `asyncio.sleep()` | **Tidak mengganggu.** Jeda tick tetap ≈ 0,1 detik. | `await asyncio.sleep()` mendaftarkan timer lalu **menyerahkan giliran** ke event loop, sehingga coroutine lain (termasuk pengukur) tetap bisa berjalan. |
| 2. `time.sleep()` | **Mengganggu.** Satu tick terlambat sekitar 0,6 detik (sama dengan lama `sleep`). | `time.sleep()` adalah fungsi *blocking*. Ia menahan satu-satunya thread, dan karena tidak ada `await`, event loop tidak punya kesempatan beralih ke coroutine lain. |
| 3. CPU-bound | **Mengganggu.** Satu tick terlambat sepanjang lama komputasi. | Loop perkalian terus memakai CPU tanpa titik `await`. Dalam *cooperative multitasking*, perpindahan giliran **hanya** terjadi di `await`, jadi tidak ada yang bisa menyela. |

**Mengapa bisa begitu?** Asyncio memakai *cooperative multitasking* dalam **satu thread**. Event loop tidak bisa "memotong" coroutine seperti OS memotong thread (*preemptive*). Selama sebuah coroutine belum mencapai `await`, seluruh program menunggu. Satu coroutine "nakal" cukup untuk membuat semua coroutine lain macet.

**Perbedaan penting antara (2) dan (3):**
- `time.sleep()` hanya *menunggu*, CPU sebenarnya nganggur. Solusinya mengganti dengan `await asyncio.sleep()` (atau library async), atau bila terpaksa memanggil fungsi blocking, menjalankannya di thread terpisah dengan `asyncio.to_thread()` / `loop.run_in_executor()`.
- Operasi CPU-bound benar-benar memakai CPU, jadi tidak bisa diselesaikan dengan async. Solusinya memindahkannya ke proses lain (`ProcessPoolExecutor` lewat `run_in_executor`), sama seperti pada Pertemuan 3 (multiprocessing).

Ini sejalan dengan *decision matrix* di materi: asyncio unggul untuk I/O-bound, sedangkan CPU-bound lebih cocok dengan multiprocessing.

---
## Soal 3 — Producer-Multi-Consumer dengan Prioritas

**Perintah:** Modifikasi contoh `asyncio.Queue` (bagian 6.4) agar setiap item punya "prioritas" (angka 1–3), dan consumer memproses item berprioritas tinggi (**angka lebih kecil**) lebih dulu. Hint: `asyncio.PriorityQueue`.

**Hal yang perlu diubah dari contoh di materi:**
1. `asyncio.Queue` → `asyncio.PriorityQueue`. Antrean ini selalu mengeluarkan item dengan nilai **terkecil** lebih dulu.
2. Item dibuat berbentuk **tuple**: `(prioritas, nomor_urut, nama_item, waktu_masuk)`. Tuple dibandingkan elemen demi elemen, jadi `prioritas` harus di depan. `nomor_urut` ditaruh kedua sebagai *tie-breaker* supaya item dengan prioritas sama tetap keluar sesuai urutan masuk (FIFO).
3. **Sinyal selesai tidak boleh lagi `None`.** `None` tidak bisa dibandingkan dengan tuple, sehingga akan menimbulkan `TypeError`. Gantinya kita pakai tuple dengan prioritas **99** (lebih besar dari semua prioritas asli), sehingga sinyal selesai selalu keluar paling akhir, setelah semua item asli habis diproses.
4. Producer dibuat lebih cepat daripada consumer supaya antrean sempat terisi. Efek prioritas baru terlihat bila ada beberapa item yang sedang menunggu bersamaan.

In [ ]:
JUMLAH_ITEM = 15
JUMLAH_CONSUMER = 3
PRIORITAS_SELESAI = 99          # sinyal selesai: prioritas "terburuk" agar keluar paling akhir

log_proses = []                 # (nama_consumer, nama_item, prioritas, waktu_tunggu)


async def producer(queue, jumlah_item):
    for i in range(jumlah_item):
        prioritas = random.randint(1, 3)                  # 1 = paling penting
        await asyncio.sleep(random.uniform(0.01, 0.03))   # producer CEPAT
        item = (prioritas, i, f"item-{i}", time.perf_counter())
        await queue.put(item)
        print(f"[Producer] menghasilkan: item-{i} (prioritas {prioritas})")
    # Kirim sinyal selesai untuk setiap consumer
    for k in range(JUMLAH_CONSUMER):
        await queue.put((PRIORITAS_SELESAI, jumlah_item + k, None, 0))


async def consumer(queue, nama):
    while True:
        prioritas, _, nama_item, waktu_masuk = await queue.get()
        if nama_item is None:                             # sinyal selesai
            queue.task_done()
            break
        waktu_tunggu = time.perf_counter() - waktu_masuk
        await asyncio.sleep(random.uniform(0.1, 0.2))     # consumer LAMBAT
        print(f"    [{nama}] memproses: {nama_item} (prioritas {prioritas})")
        log_proses.append((nama, nama_item, prioritas, waktu_tunggu))
        queue.task_done()


random.seed(42)                                           # supaya hasil bisa direproduksi
queue = asyncio.PriorityQueue(maxsize=10)

await asyncio.gather(
    producer(queue, JUMLAH_ITEM),
    *[consumer(queue, f"Consumer-{i}") for i in range(JUMLAH_CONSUMER)],
)

print(f"\nSemua {len(log_proses)} item sudah diproduksi dan dikonsumsi oleh {JUMLAH_CONSUMER} consumer.")

Untuk membuktikan bahwa prioritas benar-benar bekerja, kita hitung **rata-rata waktu tunggu di antrean** untuk tiap prioritas. Item berprioritas 1 seharusnya menunggu paling singkat.

In [ ]:
tunggu_per_prioritas = {1: [], 2: [], 3: []}
for _, _, prioritas, waktu_tunggu in log_proses:
    tunggu_per_prioritas[prioritas].append(waktu_tunggu)

print(f"{'Prioritas':>9} | {'Jumlah item':>11} | {'Rata-rata waktu tunggu':>24}")
print("-" * 50)
for prioritas in (1, 2, 3):
    daftar = tunggu_per_prioritas[prioritas]
    rata = f"{statistics.mean(daftar):.3f} detik" if daftar else "-"
    print(f"{prioritas:>9} | {len(daftar):>11} | {rata:>24}")

### Analisis Soal 3

- Pada log di atas, ketika beberapa item sedang menunggu di antrean, consumer **tidak lagi mengambil berdasarkan urutan masuk**, melainkan mengambil item dengan prioritas terkecil lebih dulu. Item berprioritas 1 bisa "menyalip" item berprioritas 3 yang masuk lebih awal.
- Tabel rata-rata waktu tunggu mendukung hal itu: item berprioritas 1 cenderung menunggu paling singkat, dan item berprioritas 3 paling lama. Angkanya bisa berbeda antar-run karena produksi dan pemrosesan memakai waktu acak.
- **Catatan penting:** prioritas hanya berpengaruh pada item yang **sedang berada bersamaan di antrean**. Kalau consumer lebih cepat daripada producer, antrean hampir selalu kosong dan item diproses langsung begitu masuk, sehingga prioritas tidak terlihat. Karena itu producer dibuat lebih cepat dari consumer.
- `maxsize=10` tetap memberi *backpressure*: bila antrean penuh, `await queue.put()` membuat producer menunggu.
- Pelajaran dari bug yang mudah terjadi: sinyal selesai `None` dari contoh asli akan membuat `PriorityQueue` error saat membandingkan `None` dengan tuple. Dengan prioritas 99, sinyal selesai juga **dijamin diproses terakhir**, jadi tidak ada item asli yang tertinggal.

---
## Soal 4 — Refleksi Singkat (maks. 150 kata)

**Perintah:** Bandingkan pengalaman menulis kode paralel dengan `threading` (Pertemuan 2), `multiprocessing` (Pertemuan 3), dan `asyncio` (pertemuan ini) untuk kasus I/O-bound. Mana yang paling mudah dipahami dan mengapa? Berikan alasan berbasis pengalaman coding, bukan hanya teori.

**Jawaban:**

Menurut saya `asyncio` paling mudah dipahami untuk kasus I/O-bound, meskipun awalnya paling membingungkan. Pada `threading`, kodenya terlihat sederhana, tetapi saya harus selalu memikirkan `Lock`, karena OS bisa memindahkan giliran thread kapan saja. Bug seperti race condition pada `counter += 1` sulit dilacak karena tidak selalu muncul. Pada `multiprocessing`, saya harus memikirkan serialisasi data dan komunikasi antarproses, padahal tugas I/O-bound tidak butuh banyak core. Pada `asyncio`, perpindahan giliran hanya terjadi di titik `await`, sehingga alur program bisa saya telusuri dan hampir tidak butuh `Lock`. Kesulitannya ada pada aturan baru: harus memakai library async, dan satu pemanggilan blocking seperti `time.sleep()` bisa menghentikan semua coroutine, seperti yang terlihat pada Soal 2. Setelah aturan itu dipahami, perilaku programnya jauh lebih mudah diprediksi.